In [1]:
import sys
sys.path.append('../')
from source import Process
import pandas as pd
import re
import json
import numpy as np

/home/tiago/anaconda3/envs/databases/lib/python3.10/site-packages/Bio/Application/__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


In [2]:
NcrdClusters = Process.ClusterCounter("../data/filtered/ncrd95.21032025.tagged.fasta")
ResfinderClusters = Process.ClusterCounter("../data/filtered/resfinder.21032025.tagged.fasta")
CardClusters = Process.ClusterCounter("../data/filtered/card.21032025.tagged.fasta")
MegaresClusters = Process.ClusterCounter("../data/filtered/megares.21032025.tagged.fasta")
HmdClusters = Process.ClusterCounter("../data/filtered/hmd.21032025.tagged.fasta")
NdaroClusters = Process.ClusterCounter("../data/filtered/ndaro.21032025.tagged.fasta")

Program: CD-HIT, V4.8.1 (+OpenMP), Apr 24 2025, 22:00:32
Command: cd-hit -i
         ../data/filtered/ncrd95.21032025.tagged.fasta -o
         ../data/filtered/ncrd95.21032025.tagged.fasta.out -c 1
         -T 12 -p 1 -sc 1 -sf 1 -aL 0.8

Started: Sun May  3 09:36:35 2026
                            Output                              
----------------------------------------------------------------
total seq: 34008
longest and shortest : 2975 and 21
Total letters: 11158353
Sequences have been sorted

Approximated minimal memory consumption:
Sequence        : 15M
Buffer          : 12 X 17M = 204M
Table           : 2 X 65M = 131M
Miscellaneous   : 0M
Total           : 352M

Table limit with the given memory limit:
Max number of representatives: 883058
Max number of word counting entries: 55961112

# comparing sequences from          0  to       2429
..---------- new table with     2429 representatives
# comparing sequences from       2429  to       4684
----------   2155 remaining seque

In [3]:
ClusterLimits = pd.DataFrame({
    "Database": ["NCRD", "Resfinder", "CARD", "MEGARES", "HMD", "NDARO"],
    "100%": [NcrdClusters[0][0], ResfinderClusters[0][0], CardClusters[0][0], MegaresClusters[0][0], HmdClusters[0][0], NdaroClusters[0][0]],
    "95%":  [NcrdClusters[0][1], ResfinderClusters[0][1], CardClusters[0][1], MegaresClusters[0][1], HmdClusters[0][1], NdaroClusters[0][1]],
    "90%":  [NcrdClusters[0][2], ResfinderClusters[0][2], CardClusters[0][2], MegaresClusters[0][2], HmdClusters[0][2], NdaroClusters[0][2]],
    "85%":  [NcrdClusters[0][3], ResfinderClusters[0][3], CardClusters[0][3], MegaresClusters[0][3], HmdClusters[0][3], NdaroClusters[0][3]],
    "80%":  [NcrdClusters[0][4], ResfinderClusters[0][4], CardClusters[0][4], MegaresClusters[0][4], HmdClusters[0][4], NdaroClusters[0][4]],  
    "75%":  [NcrdClusters[0][5], ResfinderClusters[0][5], CardClusters[0][5], MegaresClusters[0][5], HmdClusters[0][5], NdaroClusters[0][5]],
    "70%":  [NcrdClusters[0][6], ResfinderClusters[0][6], CardClusters[0][6], MegaresClusters[0][6], HmdClusters[0][6], NdaroClusters[0][6]]
})
ClusterLimits = ClusterLimits.set_index("Database").T
ClusterLimits.columns.name = None
ClusterLimits.to_csv("../data/processed/cdhitclusters.defaultsettings.csv", index=True, sep = "\t")

In [4]:
PairWiseAlignment = pd.read_csv(
    "../data/filtered/AllDatabases.Paiwise.cov80.maxseq1.tsv", 
    sep = "\t",
    skipinitialspace=True, 
    header=None,
    names = "qseqid sseqid pident length qlen slen qstart qend sstart send evalue bitscore ppos full_qseq full_sseq".split(" ")
)

PairWiseAlignment["qseqtag"] = PairWiseAlignment["qseqid"].str.split("|").str[1]
PairWiseAlignment["sseqtag"] = PairWiseAlignment["sseqid"].str.split("|").str[1]
PairWiseAlignment["qseqid"] = PairWiseAlignment["qseqid"].str.split("|").str[0]
PairWiseAlignment["sseqid"] = PairWiseAlignment["sseqid"].str.split("|").str[0]
PairWiseAlignment = PairWiseAlignment.loc[PairWiseAlignment.pident >= 95]
PairWiseAlignment.replace({"RESFINDER":"ResFinder", "MEGARES":"MegaRes","HMDARG":"HMDARG-DB"}, inplace = True)
BlastPairWiseAlignmentPivoted = PairWiseAlignment.pivot_table(index="qseqtag", columns="sseqtag", values="pident", aggfunc="count").fillna(0)
BlastPairWiseAlignmentPivoted.index.name = None
BlastPairWiseAlignmentPivoted.columns.name = None
BlastPairWiseAlignmentPivoted.to_csv("../data/processed/BlastPairWiseAlignmentPivoted.cov80.maxseq1.csv", index=True, sep = "\t")


In [5]:
BlastPairWiseAlignmentPivoted

,CARD,HMD,MegaRes,NCRD,NDARO,ResFinder
CARD,25.0,2866.0,1774.0,84.0,1285.0,0.0
HMD,4018.0,6921.0,1283.0,1988.0,264.0,6.0
MegaRes,4786.0,1373.0,164.0,67.0,241.0,196.0
NCRD,2038.0,3564.0,231.0,4963.0,65.0,1.0
NDARO,5771.0,985.0,302.0,40.0,69.0,15.0
ResFinder,2254.0,327.0,336.0,0.0,18.0,10.0


In [6]:
PairWiseAlignment

,qseqid,sseqid,pident,length,qlen,slen,qstart,qend,sstart,send,evalue,bitscore,ppos,full_qseq,full_sseq,qseqtag,sseqtag
0,CARD_0,MEGARES_1086,100.0,296,296,296,1,296,1,296,5.450000e-215,587.0,100.0,MKAYFIAILTLFTCIATVVRAQQMSELENRIDSLLNGKKATVGIAV...,MKAYFIAILTLFTCIATVVRAQQMSELENRIDSLLNGKKATVGIAV...,CARD,MegaRes
1,CARD_1,HMD_829,100.0,286,286,286,1,286,1,286,3.770000e-201,551.0,100.0,MRYIRLCIISLLAALPLAVHASPQPLEQIKQSESQLSGRVGMIEMD...,MRYIRLCIISLLAALPLAVHASPQPLEQIKQSESQLSGRVGMIEMD...,CARD,HMD
2,CARD_2,NCRD_0,100.0,164,164,164,1,164,1,164,3.050000e-117,328.0,100.0,MIGLIVARSKNNVIGKNGNIPWKIKGEQKQFRELTTGNVVIMGRKS...,MIGLIVARSKNNVIGKNGNIPWKIKGEQKQFRELTTGNVVIMGRKS...,CARD,NCRD
3,CARD_3,HMD_1374,100.0,291,291,291,1,291,1,291,5.800000e-203,556.0,100.0,MVTKRVQRMMFAAAACIPLLLGSAPLYAQTSAVQQKLAALEKSSGG...,MVTKRVQRMMFAAAACIPLLLGSAPLYAQTSAVQQKLAALEKSSGG...,CARD,HMD
4,CARD_4,HMD_247,100.0,270,270,270,1,270,1,270,2.760000e-195,535.0,100.0,MELPNIMHPVAKLSTALAAALMLSGCMPGEIRPTIGQQMETGDQRF...,MELPNIMHPVAKLSTALAAALMLSGCMPGEIRPTIGQQMETGDQRF...,CARD,HMD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74155,RESFINDER_2950,HMD_5549,100.0,283,283,283,1,283,1,283,1.220000e-200,549.0,100.0,MLRSRVTVFGILNLTEDSFFDESRRLDPAGAVTAAIEMLRVGSDVV...,MLRSRVTVFGILNLTEDSFFDESRRLDPAGAVTAAIEMLRVGSDVV...,ResFinder,HMD
74156,RESFINDER_2951,MEGARES_4771,100.0,275,275,275,1,275,1,275,2.870000e-195,535.0,100.0,MVTVFGILNLTEDSFFDESRRLDPAGAVTAAIEMLRVGSDVVDVGP...,MVTVFGILNLTEDSFFDESRRLDPAGAVTAAIEMLRVGSDVVDVGP...,ResFinder,MegaRes
74157,RESFINDER_2952,HMD_8396,100.0,279,279,279,1,279,1,279,7.040000e-198,542.0,100.0,MVTVFGILNLTEDSFFDESRRLDPAGAVTAAIEMLRVGSDVVDVGP...,MVTVFGILNLTEDSFFDESRRLDPAGAVTAAIEMLRVGSDVVDVGP...,ResFinder,HMD
74158,RESFINDER_2953,MEGARES_4770,100.0,279,279,279,1,279,1,279,1.730000e-198,543.0,100.0,MVTVFGILNLTEDSFFDESRRLDPAGAVTAAIEMLRVGSDVVDVGP...,MVTVFGILNLTEDSFFDESRRLDPAGAVTAAIEMLRVGSDVVDVGP...,ResFinder,MegaRes


In [7]:
CardIndex = pd.read_csv(
    "../data/raw/CARD_Index.tsv",
    sep = "\t"
)
CardDict = CardIndex.drop_duplicates(subset="ARO Accession", keep='first').set_index("ARO Accession").to_dict(orient="index")
## Duplicates
CardIndex.value_counts("ARO Accession")[:10]

ARO Accession
ARO:3002118    2
ARO:3003900    2
ARO:3003893    2
ARO:3000506    2
ARO:3003170    2
ARO:3006379    1
ARO:3006378    1
ARO:3006377    1
ARO:3006376    1
ARO:3006375    1
Name: count, dtype: int64

In [8]:
NDAROIndex = pd.read_csv(
    "../data/raw/NDARO_Index.tsv", 
    sep="\t")
NdaroDict = NDAROIndex.dropna(subset=["RefSeq protein"]).drop_duplicates(subset="RefSeq protein", keep='first').set_index("RefSeq protein").to_dict(orient="index")
NDAROIndex.value_counts("RefSeq protein")[:10]

RefSeq protein
WP_000090315.1    61
WP_000918664.1    44
WP_003112576.1    32
WP_001281271.1    24
WP_003436174.1    24
WP_001281881.1    23
WP_001281243.1    23
WP_001212189.1    23
WP_003094139.1    22
WP_005693446.1    22
Name: count, dtype: int64

In [9]:
SCHEMAS = {
    "CARD": {
        "AROSplitPoint": 3,
        "IndexInfo": CardDict
    },
    "NDARO": {
        "AccSplitPoint": 0,
        "IndexInfo": NdaroDict
    },
    "MEGARES": {
        "DrugClassSplitPoint": 3,
        "NameSplitPoint": -1
    },
    "HMD": {
        "DrugClassSplitPoint": 4,
        "NameSplitPoint": 5
    },
    "NCRD": {
        "DrugClassSplitPoint": 4,
        "NameSplitPoint": 2
    },
    "RESFINDER": {
        "DrugClassSplitPoint": 2,
        "NameSplitPoint": 1
    }
}

In [10]:
MetaDict = Process.CreateMetadataFile(
    "../data/filtered/AllDatabases.21032025.tagged.fasta_ids",
    SCHEMAS
    )


In [11]:
MetaDict

{'CARD_0': {'Drug Class': 'cephalosporin', 'Name': 'CblA-1'},
 'CARD_1': {'Drug Class': 'cephalosporin;penicillin beta-lactam',
  'Name': 'SHV-52'},
 'CARD_2': {'Drug Class': 'diaminopyrimidine antibiotic', 'Name': 'dfrF'},
 'CARD_3': {'Drug Class': 'cephalosporin', 'Name': 'CTX-M-130'},
 'CARD_4': {'Drug Class': 'carbapenem;cephalosporin;penicillin beta-lactam',
  'Name': 'NDM-6'},
 'CARD_5': {'Drug Class': 'carbapenem;cephalosporin;penicillin beta-lactam',
  'Name': 'ACT-35'},
 'CARD_6': {'Drug Class': 'penicillin beta-lactam', 'Name': 'CARB-5'},
 'CARD_7': {'Drug Class': 'lincosamide antibiotic;macrolide antibiotic;streptogramin A antibiotic;streptogramin B antibiotic;streptogramin antibiotic',
  'Name': 'Erm(34)'},
 'CARD_8': {'Drug Class': 'cephalosporin;monobactam;penicillin beta-lactam',
  'Name': 'TEM-126'},
 'CARD_9': {'Drug Class': 'cephalosporin;penicillin beta-lactam',
  'Name': 'LRA-12'},
 'CARD_10': {'Drug Class': 'cephalosporin;monobactam;penicillin beta-lactam',
  'Name

In [12]:
PairWiseAlignment = pd.read_csv(
    "../data/filtered/AllDatabases.Paiwise.cov80.maxseq1.tsv", 
    sep = "\t",
    skipinitialspace=True, 
    header=None,
    names = "qseqid sseqid pident length qlen slen qstart qend sstart send evalue bitscore ppos full_qseq full_sseq".split(" ")
)
PairWiseAlignment["qcov"] = np.round((PairWiseAlignment["qend"] - PairWiseAlignment["qstart"] + 1) / (PairWiseAlignment["qlen"]) * 100, 2)
PairWiseAlignment["scov"] = np.round((PairWiseAlignment["send"] - PairWiseAlignment["sstart"] + 1) / (PairWiseAlignment["slen"]) * 100, 2)
PairWiseAlignment["qseqtag"] = PairWiseAlignment["qseqid"].str.split("|").str[1]
PairWiseAlignment["sseqtag"] = PairWiseAlignment["sseqid"].str.split("|").str[1]
PairWiseAlignment["qseqid"] = PairWiseAlignment["qseqid"].str.split("|").str[0]
PairWiseAlignment["sseqid"] = PairWiseAlignment["sseqid"].str.split("|").str[0]

In [13]:
PairWiseDictSource = PairWiseAlignment[["qseqid","full_qseq"]].set_index("qseqid").to_dict()["full_qseq"]
PairWiseDictTarget = PairWiseAlignment[["sseqid","full_sseq"]].set_index("sseqid").to_dict()["full_sseq"]

In [14]:
PairWiseDictTarget

{'MEGARES_1086': 'MKAYFIAILTLFTCIATVVRAQQMSELENRIDSLLNGKKATVGIAVWTDKGDMLRYNDHVHFPLLSVFKFHVALAVLDKMDKQSISLDSIVSIKASQMPPNTYSPLRKKFPDQDFTITLRELMQYSISQSDNNACDILIEYAGGIKHINDYIHRLSIDSFNLSETEDGMHSSFEAVYRNWSTPSAMVRLLRTADEKELFSNKELKDFLWQTMIDTETGANKLKGMLPAKTVVGHKTGSSDRNADGMKTADNDAGLVILPDGRKYYIAAFVMDSYETDEDNANIIARISRMVYDAMR',
 'HMD_829': 'MRYIRLCIISLLAALPLAVHASPQPLEQIKQSESQLSGRVGMIEMDLASGRTLTAWRADERFPMISTFKVVLCGAVLARVDAGDEQLERKIHYRQQDLVDYSPVSEKHLADGMTVGELCAAAITMSDNSAANLLLAIVGGPAGLTAFLRQIGDNVTRLDRWETELNEALPGDARDTTTPASMAATLRKLLTSQRLSARSQRQLLQWMVDDRVAGPLIRSVLPAGWFIADKTGAGERGARGIVALLGPNNKAERIVVIYLRDTPASMAERNQQIAGIGAALIEHWQR',
 'NCRD_0': 'MIGLIVARSKNNVIGKNGNIPWKIKGEQKQFRELTTGNVVIMGRKSYEEIGHPLPNRMNIVVSTTTEYQGDNLVSVKSLEDALLLAKGRDVYISGGYGLFKEALQIVDKMYITEVDLNIEDGDTFFPEFDINDFEVLIGETLGEEVKYTRTFYVRKNELSRFWI',
 'HMD_1374': 'MVTKRVQRMMFAAAACIPLLLGSAPLYAQTSAVQQKLAALEKSSGGRLGVALIDTADNTQVLYRGDERFPMCSTSKVMAAAAVLKQSETQKQLLNQPVEIKPADLVNYNPIAEKHVNGTMTLAELSAAALQYSDNTAMNKLIAQLGGPGGVTAFARAIGDETFRLDRTEPTLNTAIPGDPRDTTTPRA

In [15]:
def MergeSequences(base, override1, override2):
    for key, value in base.items():
        try:     
            base[key].update({"Sequence": override1[key]})
            try:
                base[key].update({"Sequence": override2[key]})
            except:
                pass
        except:
            pass
    return base

In [16]:
MergeSequences(MetaDict, PairWiseDictSource, PairWiseDictTarget)

{'CARD_0': {'Drug Class': 'cephalosporin',
  'Name': 'CblA-1',
  'Sequence': 'MKAYFIAILTLFTCIATVVRAQQMSELENRIDSLLNGKKATVGIAVWTDKGDMLRYNDHVHFPLLSVFKFHVALAVLDKMDKQSISLDSIVSIKASQMPPNTYSPLRKKFPDQDFTITLRELMQYSISQSDNNACDILIEYAGGIKHINDYIHRLSIDSFNLSETEDGMHSSFEAVYRNWSTPSAMVRLLRTADEKELFSNKELKDFLWQTMIDTETGANKLKGMLPAKTVVGHKTGSSDRNADGMKTADNDAGLVILPDGRKYYIAAFVMDSYETDEDNANIIARISRMVYDAMR'},
 'CARD_1': {'Drug Class': 'cephalosporin;penicillin beta-lactam',
  'Name': 'SHV-52',
  'Sequence': 'MRYIRLCIISLLAALPLAVHASPQPLEQIKQSESQLSGRVGMIEMDLASGRTLTAWRADERFPMISTFKVVLCGAVLARVDAGDEQLERKIHYRQQDLVDYSPVSEKHLADGMTVGELCAAAITMSDNSAANLLLAIVGGPAGLTAFLRQIGDNVTRLDRWETELNEALPGDARDTTTPASMAATLRKLLTSQRLSARSQRQLLQWMVDDRVAGPLIRSVLPAGWFIADKTGAGERGARGIVALLGPNNKAERIVVIYLRDTPASMAERNQQIAGIGAALIEHWQR'},
 'CARD_2': {'Drug Class': 'diaminopyrimidine antibiotic',
  'Name': 'dfrF',
  'Sequence': 'MIGLIVARSKNNVIGKNGNIPWKIKGEQKQFRELTTGNVVIMGRKSYEEIGHPLPNRMNIVVSTTTEYQGDNLVSVKSLEDALLLAKGRDVYISGGYGLFKEALQIVDKMYITEVDLNIEDGDTFFPEFDINDFEVLIGE

In [17]:
def CreateGeneralClass(base):
    for key, value in base.items():
        DrugClass = base[key]["Drug Class"]
        DrugClass = DrugClass.replace("antibiotic", "").lower().strip()
        base[key].update({"Drug Class": DrugClass.rstrip('s')})

        if "cephalosporin" in DrugClass:
            base[key].update({"Drug Class": "beta-lactam"})
        if "carbapenem" in DrugClass:
            base[key].update({"Drug Class": "beta-lactam"})
        if "penicillin" in DrugClass:
            base[key].update({"Drug Class": "beta-lactam"})
        if "beta_lactam" in DrugClass:
            base[key].update({"Drug Class": "beta-lactam"})
        if "betalactams" in DrugClass:
            base[key].update({"Drug Class": "beta-lactam"})
        if "penicillin" in DrugClass:
            base[key].update({"Drug Class": "beta-lactam"})
        if "phosphonic acid" in DrugClass:
            base[key].update({"Drug Class": "phosphonic acid"})

        Macrolide = bool(re.search(rf'\b{re.escape("macrolide")}\b', DrugClass))
        Lincosamide = bool(re.search(rf'\b{re.escape("lincosamide")}\b', DrugClass))
        Streptogramin = bool(re.search(rf'\b{re.escape("streptogramin")}\b', DrugClass))
        if Macrolide and Lincosamide and Streptogramin:
            base[key].update({"Drug Class": "MLS"})


    return base

In [18]:
"Tiagos".strip('s')

'Tiago'

In [19]:
MetaDict = CreateGeneralClass(MetaDict)

In [20]:
MetaDict

{'CARD_0': {'Drug Class': 'beta-lactam',
  'Name': 'CblA-1',
  'Sequence': 'MKAYFIAILTLFTCIATVVRAQQMSELENRIDSLLNGKKATVGIAVWTDKGDMLRYNDHVHFPLLSVFKFHVALAVLDKMDKQSISLDSIVSIKASQMPPNTYSPLRKKFPDQDFTITLRELMQYSISQSDNNACDILIEYAGGIKHINDYIHRLSIDSFNLSETEDGMHSSFEAVYRNWSTPSAMVRLLRTADEKELFSNKELKDFLWQTMIDTETGANKLKGMLPAKTVVGHKTGSSDRNADGMKTADNDAGLVILPDGRKYYIAAFVMDSYETDEDNANIIARISRMVYDAMR'},
 'CARD_1': {'Drug Class': 'beta-lactam',
  'Name': 'SHV-52',
  'Sequence': 'MRYIRLCIISLLAALPLAVHASPQPLEQIKQSESQLSGRVGMIEMDLASGRTLTAWRADERFPMISTFKVVLCGAVLARVDAGDEQLERKIHYRQQDLVDYSPVSEKHLADGMTVGELCAAAITMSDNSAANLLLAIVGGPAGLTAFLRQIGDNVTRLDRWETELNEALPGDARDTTTPASMAATLRKLLTSQRLSARSQRQLLQWMVDDRVAGPLIRSVLPAGWFIADKTGAGERGARGIVALLGPNNKAERIVVIYLRDTPASMAERNQQIAGIGAALIEHWQR'},
 'CARD_2': {'Drug Class': 'diaminopyrimidine',
  'Name': 'dfrF',
  'Sequence': 'MIGLIVARSKNNVIGKNGNIPWKIKGEQKQFRELTTGNVVIMGRKSYEEIGHPLPNRMNIVVSTTTEYQGDNLVSVKSLEDALLLAKGRDVYISGGYGLFKEALQIVDKMYITEVDLNIEDGDTFFPEFDINDFEVLIGETLGEEVKYTRTFYVRKNELSRFWI'},
 'CARD_3':

In [21]:
Classes = pd.Series([MetaDict[key]["Drug Class"] for key, value in MetaDict.items()])

In [22]:
Classes.value_counts().head(10)

beta-lactam        25793
multidrug          13300
glycopeptide        8142
aminoglycoside      4926
bacitracin          4228
tetracycline        2923
MLS                 2577
na                  1290
chloramphenicol     1137
quinolone           1037
Name: count, dtype: int64

In [23]:
with open("../data/processed/MetaDict.json", "w+") as json_file  :
    json.dump(MetaDict, json_file, indent=4)